# 📊 Análisis Exploratorio de Ventas — Recorrido Guiado

Este notebook acompaña al pipeline principal (`scripts/run_pipeline.py`) y te permite **explorar cada paso de forma interactiva**.

**Contenido:**
1. Generación y carga de datos sintéticos
2. KPIs de negocio
3. Estadística descriptiva por categoría
4. Series temporales y estacionalidad
5. Correlaciones
6. Segmentación de clientes (RFM + K-Means)

> 💡 Ejecuta las celdas en orden con `Shift + Enter`.

In [ ]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al path para importar el paquete
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from retail_analytics import config
from retail_analytics.data.generator import generar_ventas
from retail_analytics.data.cleaner import agregar_variables, limpiar_ventas
from retail_analytics.analysis.descriptive import calcular_kpis, imprimir_kpis, resumen_por_categoria, top_productos
from retail_analytics.analysis.temporal import serie_mensual, media_movil, descomposicion_estacional
from retail_analytics.analysis.segmentation import calcular_rfm, escalar_features, ajustar_kmeans, nombrar_segmentos
from retail_analytics.visualization.style import aplicar_estilo
from retail_analytics.visualization.static_plots import (
    grafico_histograma,
    grafico_serie_temporal,
    grafico_mapa_calor,
    grafico_clusters_2d,
)

aplicar_estilo()
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
print('✅ Entorno listo — versión del paquete:', __import__('retail_analytics').__version__)

## 1️⃣ Datos sintéticos

Generamos ~8 000 transacciones con estacionalidad anual (Navidad), semanal (fines de semana) y clientes fieles tipo Pareto. La semilla fija garantiza reproducibilidad.

In [ ]:
ventas_crudas = generar_ventas(n_transacciones=8000, n_clientes=1000)
ventas = agregar_variables(limpiar_ventas(ventas_crudas))

print(f'Filas: {len(ventas):,} | Columnas: {len(ventas.columns)}')
ventas.head()

## 2️⃣ KPIs de negocio

In [ ]:
kpis = calcular_kpis(ventas)
print(imprimir_kpis(kpis))

## 3️⃣ ¿Qué categorías generan más ingresos?

In [ ]:
resumen_por_categoria(ventas)

In [ ]:
top_productos(ventas, n=10)

### Distribución del ticket
Observa la asimetría positiva: pocas compras grandes elevan la media por encima de la mediana. Por eso en retail se reportan ambas.

In [ ]:
fig = grafico_histograma(ventas['total'], bins=60, titulo='Distribución del importe por transacción', etiqueta_x='Total (€)')
fig

## 4️⃣ Serie temporal mensual
La línea roja es la media móvil a 3 meses: suaviza el ruido y revela la tendencia creciente.

In [ ]:
mensual = serie_mensual(ventas)
mm3 = media_movil(mensual, ventana=3)

fig = grafico_serie_temporal(mensual, mm3, titulo='Ventas mensuales con media móvil (3 meses)')
fig

In [ ]:
# Descomposición aditiva: tendencia + estacionalidad + residuo
descomposicion = descomposicion_estacional(mensual, periodo=12)
_ = descomposicion.plot()

## 5️⃣ Correlaciones
Recuerda: correlación ≠ causalidad. El p-valor nos dice si la relación podría deberse al azar.

In [ ]:
fig = grafico_mapa_calor(ventas)
fig

## 6️⃣ Segmentación de clientes: RFM + K-Means
- **R**ecencia: días desde la última compra
- **F**recuencia: número de compras
- Valor **M**onetario: gasto total

K-Means agrupa clientes similares; luego nombramos cada clúster según su gasto medio.

In [ ]:
rfm = calcular_rfm(ventas)
X = escalar_features(rfm)
etiquetas, modelo, silueta = ajustar_kmeans(X, k=4)

rfm['segmento'] = pd.Series(etiquetas).map(nombrar_segmentos(rfm, etiquetas)).values
print(f'Silueta del clustering: {silueta}')
rfm.groupby('segmento').agg(
    clientes=('id_cliente', 'count'),
    recencia_media=('recencia_dias', 'mean'),
    frecuencia_media=('frecuencia', 'mean'),
    gasto_medio=('valor_monetario', 'mean'),
).round(1).sort_values('gasto_medio', ascending=False)

In [ ]:
fig = grafico_clusters_2d(rfm)
fig

## 🎯 Conclusiones típicas del análisis

1. **Tendencia**: las ventas crecen mes a mes (~30 % en dos años).
2. **Estacionalidad**: picos claros en noviembre-diciembre (Black Friday + Navidad) y fines de semana fuertes.
3. **Categorías**: Electrónica lidera ingresos pese a menos transacciones que Alimentación/Ropa (ticket alto).
4. **Clientes**: existe un grupo pequeño de "Campeones" que concentra gran parte del gasto → candidato a programa VIP.

---
*Siguiente paso: ejecuta `python scripts/run_pipeline.py` para generar todos los informes y gráficas automáticamente.*